%matplotlib inline

In [2]:
import os
os.environ['USE_PYGEOS'] = '0'
os.environ['PROJ_DEBUG'] = '2' # For showing logs
os.environ['GDAL_PAM_ENABLED'] = 'YES'
os.environ['ESRI_XML_PAM'] = 'YES'
#os.environ['GDAL_IGNORE_COG_LAYOUT_BREAK'] = 'YES'
#os.environ['PROJ_NETWORK'] = 'ON' # Ensure this is 'ON' to get shift grids over the internet
print(os.environ['PROJ_NETWORK']) 
 
import geopandas as gpd
from osgeo import gdal
import shapely
import pyproj
import rasterio
from rasterio.warp import calculate_default_transform
#import coincident

ON


In [3]:
def fetch_worldcover(raster_fn,match_grid_da=None):
    with rasterio.open(raster_fn) as dataset:
        bounds = dataset.bounds
        bounds = rasterio.warp.transform_bounds(dataset.crs, 'EPSG:4326', *bounds)
        bbox_gdf = gpd.GeoDataFrame(geometry=[shapely.box(*bounds)],crs='EPSG:4326',index=[0])
        
    
    # Load Worldcover DEM for this bounding box (from Scott's notebook)
    gf_wc = coincident.search.search(
        dataset="worldcover",
        intersects=bbox_gdf,
    )  # Asset of interest = 'data'
    if match_grid_da is not None:
        ds_wc = coincident.io.xarray.to_dataset(
            gf_wc,
            bands=["map"],
            #aoi=bbox,
            like=match_grid_da, # Match grid of COP30 (both considered 4326)
        )
    else:
        ds_wc = coincident.io.xarray.to_dataset(
            gf_wc,
            bands=["map"],
            #aoi=bbox,
            )
    da_wc = ds_wc.isel(time=1).to_dataarray().squeeze()
    return da_wc


def common_mask(da_list,apply=False):
    """
    From a list of xarray dataarray objects sharing the same projection/extent/res, compute common mask where all input datasets have non-nan pixels
    """
    # load nan layers as numpy array
    nan_arrays = np.array([np.isnan(da.values) for da in da_list])
    common_mask = 1 - np.any(nan_arrays,axis=0)
    
    if apply:
        common_mask_da_list = [da.where(common_mask,np.nan) for da in da_list]
        return common_mask_da_list
    else:
        return common_mask

def fetch_cop30(raster_fn,match_grid_da=None):
    with rasterio.open(raster_fn) as dataset:
        bounds = dataset.bounds
        bounds = rasterio.warp.transform_bounds(dataset.crs, 'EPSG:4326', *bounds)
        bbox_gdf = gpd.GeoDataFrame(geometry=[shapely.box(*bounds)],crs='EPSG:4326',index=[0])
        
    
    # Load Worldcover DEM for this bounding box (from Scott's notebook)
    gf = coincident.search.search(
        dataset="cop30",
        intersects=bbox_gdf,
    )  # Asset of interest = 'data'
    if match_grid_da is not None:
        ds = coincident.io.xarray.to_dataset(
            gf,
            bands=["data"],
            #aoi=bbox,
            like=match_grid_da, # Match grid of COP30 (both considered 4326)
        )
    else:
        ds = coincident.io.xarray.to_dataset(
            gf,
            bands=["data"],
            #aoi=bbox,
            )
    da = ds.to_dataarray().squeeze()
    return da


def confirm_3dep_vertical(raster_fn,bare_diff_tolerance=3):
    lidar_da = rioxarray.open_rasterio(raster_fn,masked=True).squeeze()
    worldcover_da = fetch_worldcover(raster_fn,lidar_da)
    cop30_da = fetch_cop30(raster_fn,lidar_da)
    lidar_da_masked,worldcover_da_masked,cop30_da_masked = common_mask([lidar_da,worldcover_da,cop30_da],apply=True)
    dem_diff = lidar_da_masked - cop30_da_masked
    ## Mask out bare and sparse vegetation class
    bare_sparse_mask = worldcover_da_masked == 60
    dem_diff_bare = dem_diff.where(bare_sparse_mask,np.nan)
    median_diff = np.nanmedian(dem_diff_bare.values)
    print(f"Observed difference between COP30 EGM2008 and 3DEP LiDAR DSM over bareground and sparse vegetation surfaces is {median_diff:.2f} m")
    if np.abs(median_diff) <= bare_diff_tolerance:
        #this means that both COP30 and 3DEP LiDAR DSM are with respect to geoid
        print("Looks like the 3DEP height estimates are with respect to geiod, will apply vertical datum shift to return heights with respect to ellipsoid")
    else:
        #this means that 3DEP LiDAR DSM is with respect to ellipsoid
        print("Looks like the 3DEP height estimates are already with respect to ellipsoid, geoid to ellipsoid transformation will not be attempted")

In [4]:
%cd /panfs/ccds02/nobackup/people/sbhusha1/pcd/seattle

/panfs/ccds02/nobackup/people/sbhusha1/pcd/seattle


In [28]:
#seattle King-county test
raster_fn = 'seattle_3dep/King-county-21-lidar-DSM_mos.tif'

In [29]:
%time confirm_3dep_vertical(raster_fn)

Observed difference between COP30 EGM2008 and 3DEP LiDAR DSM over bareground and sparse vegetation surfaces is 2.17 m
Looks like the 3DEP height estimates are with respect to geiod, will apply vertical datum shift to return heights with respect to ellipsoid
CPU times: user 1.74 s, sys: 113 ms, total: 1.85 s
Wall time: 4.66 s


In [30]:
#Mt Baker lidar data test
raster_fn = '../mt-baker/baker_3dep/baker-DSM_mos.tif'

In [31]:
%time confirm_3dep_vertical(raster_fn)

Observed difference between COP30 EGM2008 and 3DEP LiDAR DSM over bareground and sparse vegetation surfaces is -18.24 m
Looks like the 3DEP height estimates are already with respect to ellipsoid, geoid to ellipsoid transformation will not be attempted
CPU times: user 869 ms, sys: 17 ms, total: 886 ms
Wall time: 2.19 s


## Block-2 Compute the final reprojection

In [4]:
##We attempt this in using rasterio fully, following Romain's code snippet from https://github.com/rasterio/rasterio/issues/2433#issuecomment-2786157846

In [4]:
   with rasterio.open(url) as src:        
        src_data = src.read()
        print(src_data.mean()) 
        
        # original metadata unaware of vertical reference
        src_crs3D = rasterio.crs.CRS.from_epsg('9518')
        dst_crs = rasterio.crs.CRS.from_epsg('7661')    
        transform, width, height = calculate_default_transform(src_crs3D, 
                                                               dst_crs, 
                                                               src.width, 
                                                               src.height, 
                                                               *src.bounds)
        kwargs = src.meta.copy()
        kwargs.update({
            'crs': dst_crs,
            'transform': transform,
            'width': width,
            'height': height
        })
        
        with rasterio.open('test7661.tif', 'w', **kwargs) as dst:
            dst_data = np.zeros((1,width,height))
            reproject(source=src_data,
                      destination=dst_data,
                      src_transform=src.transform,
                      src_crs=src_crs3D,
                      dst_transform=transform,
                      dst_crs=dst_crs,
                      resampling=Resampling.nearest
                     )
            
            # Expect dst_data (ellipsoid) mean to be -15 meters
            print(dst_data.mean())
            dst.write(dst_data)

NameError: name 'url' is not defined

In [12]:
#from Romain Hugonnet
def reproject_rasters(src_raster_fn,dst_raster_fn,src_srs,dst_srs,dst_res=None,resampling='cubic'):
    tolerance = 0
    resampling_mapping = {"nearest": rasterio.enums.Resampling.nearest, "bilinear": rasterio.enums.Resampling.bilinear,
                          "cubic": rasterio.enums.Resampling.cubic, "cubic_spline": rasterio.enums.Resampling.cubic_spline}
    with rasterio.open(src_raster_fn) as src:

        # Get input geotransform and array
        src_arr = src.read()
        print(f"Input raster median: {np.nanmedian(src_arr):.2f}")
        
        src_transform = src.transform
        src_nodata = src.nodata
        
        kwargs = src.meta.copy()
        # Calcul output geotransform and shape
        dst_transform, dst_width, dst_height = calculate_default_transform(src_srs,
                                                                           dst_srs,
                                                                           src.width,
                                                                           src.height,
                                                                           *src.bounds,resolution=dst_res)
    dst_shape = (1,dst_width, dst_height)
    dst_nodata = src_nodata
    dst_arr = np.zeros(dst_shape)
    
    kwargs.update({'crs':dst_srs,
              'transform': dst_transform,
              'width':dst_width,
              'height':dst_height,
              'nodata':dst_nodata,
             'profile':"BASELINE"})
    print(dst_height)
    print(dst_width)
    
    warp_opts = {"XSCALE": 1, "YSCALE": 1, "APPLY_VERTICAL_SHIFT": False}
    _ = rasterio.warp.reproject(
        src_arr,
        dst_arr,
        src_transform=src_transform,
        src_crs=src_srs,
        dst_transform=dst_transform,
        dst_crs=dst_srs,
        resampling=resampling_mapping[resampling],
        src_nodata=src_nodata,
        dst_nodata=dst_nodata,
        tolerance=tolerance,
        **warp_opts
    )
    print(f"Output raster median: {np.nanmedian(dst_arr):.2f}")
    
    with rasterio.open(dst_raster_fn, 'w', **kwargs) as dst:
        #dst.crs = dst_srs
        print(dst_srs)
        dst.write(dst_arr)



In [9]:
src_srs = pyproj.CRS.from_epsg('3857')

dst_srs = 'UTM_10N_WGS84_G2139_3D.wkt'
with open(dst_srs, 'r') as f: #open the file
    contents = f.read()
    crs = pyproj.CRS.from_string(contents)

## Reproject Baker data

In [19]:
raster_fn = '../mt-baker/baker_3dep/baker-DSM_mos.tif'
out_fn = '../mt-baker/baker_3dep/baker-DSM_mos_G2139.tif'

In [13]:
#dst_srs = pyproj.CRS.from_epsg('7912')
reproject_rasters(raster_fn,out_fn,src_srs,crs,resampling='cubic')

Input raster median: 1637.91
1446
2062
Output raster median: 1579.84
PROJCRS["WGS 84 (G2139) / UTM 10N",
    BASEGEOGCRS["WGS 84 (G2139)",
    DYNAMIC[
        FRAMEEPOCH[2016]],
    DATUM["World Geodetic System 1984 (G2139)",
        ELLIPSOID["WGS 84",6378137,298.257223563,
            LENGTHUNIT["metre",1]]],
    PRIMEM["Greenwich",0,
        ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",9754]],
    CONVERSION["UTM zone 10N",
        METHOD["Transverse Mercator",
            ID["EPSG",9807]],
        PARAMETER["Latitude of natural origin",0,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8801]],
        PARAMETER["Longitude of natural origin",-123,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8802]],
        PARAMETER["Scale factor at natural origin",0.9996,
            SCALEUNIT["unity",1],
            ID["EPSG",8805]],
        PARAMETER["False easting",500000,
            LENGTHUNIT["metre",1],
            ID["EP

In [20]:
ds = gdal.Warp(out_fn,raster_fn,srcSRS='EPSG:3857',dstSRS='UTM_10N_WGS84_G2139_3D.wkt')
ds = None

# This works for the below configuration

In [21]:
! conda list

# packages in environment at /panfs/ccds02/nobackup/people/sbhusha1/sw/miniconda3/envs/bhushan_gdal3:
#
# Name                    Version                   Build  Channel
_libgcc_mutex             0.1                 conda_forge    conda-forge
_openmp_mutex             4.5                       2_gnu    conda-forge
_sysroot_linux-64_curr_repodata_hack 3                   haa98f57_10  
affine                    2.4.0              pyhd8ed1ab_1    conda-forge
asttokens                 3.0.0           py312h06a4308_0  
attr                      2.5.1                h166bdaf_1    conda-forge
attrs                     25.3.0             pyh71513ae_0    conda-forge
aws-c-auth                0.8.0               hb88c0a9_10    conda-forge
aws-c-cal                 0.8.0                hecf86a2_2    conda-forge
aws-c-common              0.10.3               hb9d3cd8_0    conda-forge
aws-c-compression         0.3.0                hf42f96a_2    conda-forge
aws-c-event-stream        0.5.0          

## Reproject Seattle  data

In [25]:
raster_fn = 'seattle_3dep/King-county-21-lidar-DSM_mos.tif'
out_fn = 'seattle_3dep/King-county-21-lidar-DSM_mos_G2139.tif'

In [26]:
src_srs = "/panfs/ccds02/nobackup/people/sbhusha1/sw/lidar_tools/notebooks/SRS_CRS.wkt"
with open(src_srs, 'r') as f: #open the file
    contents = f.read()
    src_srs = pyproj.CRS.from_string(contents)
dst_srs = 'UTM_10N_WGS84_G2139_3D.wkt'
with open(dst_srs, 'r') as f: #open the file
    contents = f.read()
    crs = pyproj.CRS.from_string(contents)

In [27]:
src_srs

<Compound CRS: COMPOUNDCRS["test",PROJCRS["WGS 84 / Pseudo-Mercat ...>
Name: test
Axis Info [cartesian|vertical]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
- H[up]: Gravity-related height (metre)
Area of Use:
- undefined
Datum: World Geodetic System 1984
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich
Sub CRS:
- WGS 84 / Pseudo-Mercator
- NAVD88 height

In [28]:
#dst_srs = pyproj.CRS.from_epsg('7912')
reproject_rasters(raster_fn,out_fn,src_srs,crs,resampling='cubic')

Input raster median: 11.96


KeyboardInterrupt: 

In [29]:
from osgeo import gdal, gdalconst

In [30]:
input_ds = gdal.Open(raster_fn)

/panfs/ccds02/nobackup/people/sbhusha1/sw/miniconda3/envs/bhushan_gdal3/lib/python3.13/site-packages/osgeo/gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [36]:
band = input_ds.GetRasterBand(1)
input_array = band.ReadAsArray()
nodata = band.GetNoDataValue()
input_array = np.ma.masked_values(input_array, nodata)

In [ ]:
gdal.Warp()

In [41]:
src_srs = "/panfs/ccds02/nobackup/people/sbhusha1/sw/lidar_tools/notebooks/SRS_CRS.wkt"
with open(src_srs, 'r') as f: #open the file
    contents = f.read()
    src_crs = pyproj.CRS.from_string(contents)
dst_srs = 'UTM_10N_WGS84_G2139_3D.wkt'
with open(dst_srs, 'r') as f: #open the file
    contents = f.read()
    dst_crs = pyproj.CRS.from_string(contents)

In [8]:
from osgeo import gdalconst

In [9]:
resampling_mapping = {"nearest":  gdalconst.GRA_NearestNeighbour, "bilinear": gdalconst.GRA_Bilinear,
                  "cubic": gdalconst.GRA_Cubic, "cubic_spline": gdalconst.GRA_CubicSpline}
resampling = "cubic"
gdal_resampling = resampling_mapping[resampling]
tolerance = 1

In [ ]:
input_srs = 

In [43]:
warp_opts = {"XSCALE":1, "YSCALE": 1, "APPLY_VERTICAL_SHIFT": True}
opts = gdal.WarpOptions(srcSRS=src_crs.to_wkt(), dstSRS=dst_crs.to_wkt(),
                            resampleAlg=gdal_resampling, errorThreshold=tolerance, warpOptions=warp_opts)



In [5]:
raster_fn = 'seattle_3dep/King-county-21-lidar-DSM_mos.tif'
out_fn = 'seattle_3dep/King-county-21-lidar-DSM_mos_G2139_new_env.tif'

In [ ]:
warp = gdal.Warp(output_raster,input_raster,dstSRS='EPSG:4326')


In [ ]:
opts = gdal.War

In [12]:
ds = gdal.Warp(out_fn,raster_fn,
               resampleAlg=resampling_mapping["cubic"],srcSRS="/panfs/ccds02/nobackup/people/sbhusha1/sw/lidar_tools/notebooks/SRS_CRS.wkt",
               dstSRS='UTM_10N_WGS84_G2139_3D.wkt',errorThreshold=tolerance)
ds = None

In [48]:
! echo  $GDAL_PAM_ENABLED

In [37]:
gt = input_ds.GetGeoTransform()
#srs = ds_orig.GetProjection()
dst_ns = ds_orig.RasterXSize
dst_nl = ds_orig.RasterYSize
nbands = ds_orig.RasterCount
dtype = ds_orig.GetRasterBand(1).DataType

masked_array(
  data=[[--, --, --, ..., --, --, --],
        [--, --, 70.05249786376953, ..., --, --, --],
        [--, 70.07431030273438, 70.06527709960938, ..., 21.4997501373291,
         --, --],
        ...,
        [5.579999923706055, 5.604008674621582, 5.702763557434082, ...,
         --, --, --],
        [--, 5.64571475982666, 5.71999979019165, ..., --, --, --],
        [--, --, --, ..., --, --, --]],
  mask=[[ True,  True,  True, ...,  True,  True,  True],
        [ True,  True, False, ...,  True,  True,  True],
        [ True, False, False, ..., False,  True,  True],
        ...,
        [False, False, False, ...,  True,  True,  True],
        [ True, False, False, ...,  True,  True,  True],
        [ True,  True,  True, ...,  True,  True,  True]],
  fill_value=-9999.0,
  dtype=float32)

In [35]:
input_ds.nodata

AttributeError: 'Dataset' object has no attribute 'nodata'

In [ ]:
def _reproject_gdal(src_arr, src_transform, src_crs, dst_transform, dst_shape, dst_crs, resampling, src_nodata, dst_nodata):
    from osgeo import gdal, gdalconst

    resampling_mapping = {"nearest":  gdalconst.GRA_NearestNeighbour, "bilinear": gdalconst.GRA_Bilinear,
                  "cubic": gdalconst.GRA_Cubic, "cubic_spline": gdalconst.GRA_CubicSpline}

    gdal_resampling = resampling_mapping[resampling]

    def _rio_to_gdal_geotransform(gt: rio.Affine) -> tuple[float, ...]:

        dx, b, xmin, d, dy, ymax = list(gt)[:6]

        gdal_gt = (xmin, dx, 0, ymax, 0, dy)

        return gdal_gt

    # Create input raster from array shape
    drv = gdal.GetDriverByName('MEM')
    src_ds = drv.Create('', src_arr.shape[1], src_arr.shape[0], 1, gdal.GDT_Float32)
    # Create, define and set projection
    src_ds.SetProjection(src_crs.to_wkt())
    # Convert and set geotransform
    src_gt = _rio_to_gdal_geotransform(src_transform)
    src_ds.SetGeoTransform(src_gt)
    # Write array and nodata value on first band
    src_band = src_ds.GetRasterBand(1)
    src_band.WriteArray(src_arr)
    src_band.SetNoDataValue(src_nodata)

    # Create output raster from destination shape
    dst_ds = drv.Create("", dst_shape[1], dst_shape[0], 1, gdal.GDT_Float32)
    # Create output projection
    dst_ds.SetProjection(dst_crs.to_wkt())
    dst_gt = _rio_to_gdal_geotransform(dst_transform)
    dst_ds.SetGeoTransform(dst_gt)

    # Copy the raster metadata of the source to dest
    dst_ds.GetRasterBand(1).SetNoDataValue(dst_nodata)
    dst_ds.GetRasterBand(1).Fill(dst_nodata)

    # Reproject with resampling
    warp_opts = {"XSCALE":1, "YSCALE": 1, "APPLY_VERTICAL_SHIFT": True}
    opts = gdal.WarpOptions(srcSRS=src_crs.to_wkt(), dstSRS=dst_crs.to_wkt(),
                            resampleAlg=gdal_resampling, errorThreshold=tolerance, warpOptions=warp_opts)
    gdal.Warp(dst_ds, src_ds, options=opts)

    # Extract reprojected array
    array = dst_ds.GetRasterBand(1).ReadAsArray().astype("float32")
    array[array == dst_nodata] = np.nan

    return array

In [1]:
! conda list

# packages in environment at /panfs/ccds02/nobackup/people/sbhusha1/sw/miniconda3/envs/bhushan_gdal3:
#
# Name                    Version                   Build  Channel
_libgcc_mutex             0.1                 conda_forge    conda-forge
_openmp_mutex             4.5                       2_gnu    conda-forge
_sysroot_linux-64_curr_repodata_hack 3                   haa98f57_10  
affine                    2.4.0              pyhd8ed1ab_1    conda-forge
aiobotocore               2.21.1                   pypi_0    pypi
aiofiles                  24.1.0                   pypi_0    pypi
aiohappyeyeballs          2.6.1                    pypi_0    pypi
aiohttp                   3.11.16                  pypi_0    pypi
aiohttp-oauth2-client     1.0.2                    pypi_0    pypi
aiohttp-retry             2.9.1                    pypi_0    pypi
aioitertools              0.12.0                   pypi_0    pypi
aiosignal                 1.3.2                    pypi_0    pypi
annotated-t